# Intent-driven international keyword research: multilingual search intent classification

Companion notebook for the MLforSEO guide **"How to Do Intent-driven International Keyword Research"** by Lazarina Stoy.

It turns the tips from the guide into a repeatable workflow:

1. **Use regex strategically**: translate only the *filter words* for each intent category, then localise and extend them per market (not the whole keyword list).
2. **Adapt the filters per industry**: the intent stays the same, the way people express it changes (e-commerce vs SaaS vs services).
3. **Find the gaps**: see what share of each market is still unclassified, and which words keep showing up there, so you know which filters to extend next.
4. **Watch for language traps**: false friends, homographs and brand names that are also everyday words can inflate volumes in one market.
5. **Compare markets in Search Console data**: the same page can attract different intent in different countries.
6. **Export**: CSVs, plus ready-made regex strings for Data Studio (`REGEXP_MATCH`) or Google Sheets (`REGEXMATCH`).

No API keys needed. Runs in Google Colab or locally (Python 3.9+, pandas).

> The starter dictionaries below are a **starting point, not a finished product**. Have a native speaker (or an SEO localisation expert) review every market before you rely on the labels.

In [ ]:
# If you're running locally: pip install pandas
import re, unicodedata
import pandas as pd
pd.set_option('display.max_colwidth', 80)

## 1. Load your keywords

You need one row per keyword, with the **market** (country) and **language** it belongs to. Volume is optional but makes every summary more useful.

Export from Semrush, Ahrefs, Google Keyword Planner, DataForSEO or your own Search Console data, then either upload the CSV (set `USE_SAMPLE = False`) or run the example data first to see how it works.

Expected columns: `keyword`, `market`, `language`, `volume` (optional).

In [ ]:
USE_SAMPLE = True
CSV_PATH = 'keywords.csv'   # used when USE_SAMPLE = False

if USE_SAMPLE:
    # Small, made-up example set: the same running-shoe topic in four markets
    rows = [
        # keyword, market, language, volume
        ('how to choose running shoes', 'US', 'en', 2400),
        ('best running shoes for flat feet', 'US', 'en', 6600),
        ('nike vs asics running shoes', 'US', 'en', 880),
        ('buy trail running shoes', 'US', 'en', 390),
        ('running shoes near me', 'US', 'en', 9900),
        ('stride running shoes login', 'US', 'en', 210),
        ('running shoes sale', 'US', 'en', 4400),
        ('trainers for running', 'GB', 'en', 1900),
        ('best running trainers uk', 'GB', 'en', 3600),
        ('cheap running trainers', 'GB', 'en', 1300),
        ('what are the best trainers for running', 'GB', 'en', 720),
        ('running trainers review', 'GB', 'en', 880),
        ('cómo elegir zapatillas de running', 'ES', 'es', 1600),
        ('mejores zapatillas running', 'ES', 'es', 8100),
        ('comprar zapatillas running mujer', 'ES', 'es', 1900),
        ('zapatillas running baratas', 'ES', 'es', 2900),
        ('zapatillas running opiniones', 'ES', 'es', 590),
        ('zapatillas trail running', 'ES', 'es', 4400),
        ('laufschuhe test', 'DE', 'de', 6600),
        ('laufschuhe kaufen', 'DE', 'de', 2400),
        ('welche laufschuhe für anfänger', 'DE', 'de', 1300),
        ('laufschuhe günstig', 'DE', 'de', 1900),
        ('laufschuhe damen', 'DE', 'de', 9900),
        ('gift ideen für läufer', 'DE', 'de', 320),
    ]
    kw = pd.DataFrame(rows, columns=['keyword', 'market', 'language', 'volume'])
else:
    try:
        from google.colab import files  # upload from your computer in Colab
        uploaded = files.upload()
        CSV_PATH = next(iter(uploaded))
    except ImportError:
        pass
    kw = pd.read_csv(CSV_PATH)

kw['language'] = kw['language'].str.lower().str[:2]
kw['market'] = kw['market'].str.upper()
if 'volume' not in kw:
    kw['volume'] = 1
kw.head()

## 2. The filter words: translate the filters, not the keywords

This is tip 1 from the guide. Instead of translating thousands of keywords, you translate (and localise) a short list of filter words per intent category, per language.

* `INTENT_FILTERS` holds the starter words for each language.
* `MARKET_EXTRAS` adds market-specific variations on top: the same language, expressed differently (British "trainers" and "cheap" vs American "sneakers" and "sale", for example).
* `INDUSTRY_PRESETS` adds transactional words that depend on your industry, taken from the word clouds in the guide.
* `BRAND_TERMS` marks navigational (branded) searches. Put your brand and your competitors' brands here.

Matching ignores case and accents, so `como` matches `cómo` and `fur` matches `für`. Multi-word filters are allowed.

In [ ]:
INTENT_FILTERS = {
    'en': {
        'informational': ['how', 'what', 'why', 'when', 'who', 'which', 'where', 'does', 'guide', 'tutorial',
                          'learn', 'examples', 'ideas', 'tips', 'meaning', 'definition', 'is', 'are', 'can'],
        'commercial':    ['best', 'top', 'review', 'reviews', 'vs', 'versus', 'compare', 'comparison',
                          'alternative', 'alternatives', 'cheap', 'cheapest', 'affordable', 'recommended'],
        'transactional': ['buy', 'order', 'purchase', 'price', 'pricing', 'cost', 'coupon', 'discount',
                          'deal', 'deals', 'sale', 'near me', 'shop', 'store', 'for sale'],
        'navigational':  ['login', 'log in', 'sign in', 'account', 'contact', 'customer service', 'website',
                          'official site', 'app'],
    },
    'es': {
        'informational': ['como', 'por que', 'que es', 'que', 'cual', 'cuales', 'cuando', 'quien', 'donde',
                          'guia', 'tutorial', 'aprender', 'ejemplos', 'ideas', 'consejos', 'significado'],
        'commercial':    ['mejor', 'mejores', 'opiniones', 'opinion', 'review', 'comparativa', 'vs', 'alternativa',
                          'alternativas', 'barato', 'baratas', 'baratos', 'economico', 'recomendado', 'recomendadas'],
        'transactional': ['comprar', 'compra', 'precio', 'precios', 'oferta', 'ofertas', 'descuento', 'cupon',
                          'rebajas', 'tienda', 'cerca de mi'],
        'navigational':  ['login', 'iniciar sesion', 'mi cuenta', 'contacto', 'atencion al cliente', 'web oficial', 'app'],
    },
    'de': {
        'informational': ['wie', 'was', 'warum', 'wann', 'wer', 'welche', 'welcher', 'welches', 'wo', 'anleitung',
                          'tipps', 'ideen', 'beispiele', 'bedeutung', 'lernen', 'fur anfanger'],
        'commercial':    ['beste', 'besten', 'test', 'testsieger', 'vergleich', 'erfahrungen', 'bewertung',
                          'vs', 'alternative', 'gunstig', 'billig', 'empfehlung'],
        'transactional': ['kaufen', 'bestellen', 'preis', 'preise', 'kosten', 'angebot', 'rabatt', 'gutschein',
                          'sale', 'shop', 'in der nahe'],
        'navigational':  ['login', 'anmelden', 'mein konto', 'kontakt', 'kundenservice', 'offizielle seite', 'app'],
    },
    'fr': {
        'informational': ['comment', 'pourquoi', 'quand', 'qui', 'quel', 'quelle', 'quels', 'quelles', 'ou',
                          'guide', 'tutoriel', 'apprendre', 'exemples', 'idees', 'conseils', 'definition'],
        'commercial':    ['meilleur', 'meilleure', 'meilleurs', 'avis', 'test', 'comparatif', 'vs', 'alternative',
                          'pas cher', 'recommande'],
        'transactional': ['acheter', 'achat', 'commander', 'prix', 'tarif', 'promo', 'promotion', 'soldes',
                          'code promo', 'boutique', 'pres de chez moi'],
        'navigational':  ['connexion', 'se connecter', 'mon compte', 'contact', 'service client', 'site officiel', 'app'],
    },
}

# Same language, different market: add localised variations on top of the language list
MARKET_EXTRAS = {
    'GB': {'commercial': ['uk'], 'transactional': ['trainers sale', 'click and collect']},
    'US': {'transactional': ['sneakers sale']},
    'ES': {'transactional': ['envio gratis']},
    'DE': {'transactional': ['versandkostenfrei']},
}

# Transactional words that depend on the industry (from the word clouds in the guide)
INDUSTRY_PRESETS = {
    'ecommerce': ['buy', 'order', 'purchase', 'cheap', 'price', 'coupon', 'near'],
    'saas':      ['trial', 'free trial', 'demo', 'pricing', 'plans', 'sign up', 'signup', 'license', 'api', 'features'],
    'service':   ['book', 'call', 'contact', 'consultancy', 'consultation', 'quote', 'services', 'portfolio',
                  'testimonials', 'demo', 'pricing'],
}
INDUSTRY = 'ecommerce'   # 'ecommerce', 'saas', 'service' or None

# Your brand + competitor brands = navigational intent
BRAND_TERMS = ['stride', 'nike', 'asics', 'adidas', 'hoka']

# When a keyword matches several categories, which one wins?
# (The guide explains why you should decide this up front.)
PRIORITY = ['transactional', 'commercial', 'navigational', 'informational']

## 3. Build the classifier

In [ ]:
def normalise(text):
    """Lowercase and strip accents, so filters match however people type."""
    text = unicodedata.normalize('NFKD', str(text).lower())
    return ''.join(ch for ch in text if not unicodedata.combining(ch)).replace('ß', 'ss')

def to_regex(words):
    """One case-insensitive, whole-word regex for a list of filter words."""
    words = sorted({normalise(w) for w in words if w}, key=len, reverse=True)
    if not words:
        return None
    return r'(?:^|\b)(?:' + '|'.join(re.escape(w).replace(r'\ ', r'\s+') for w in words) + r')(?:\b|$)'

def build_filters(language, market, industry=INDUSTRY, brands=BRAND_TERMS):
    base = {k: list(v) for k, v in INTENT_FILTERS.get(language, {}).items()}
    for intent, extra in MARKET_EXTRAS.get(market, {}).items():
        base.setdefault(intent, []).extend(extra)
    if industry and language == 'en':          # presets are in English; translate them for other languages
        base.setdefault('transactional', []).extend(INDUSTRY_PRESETS[industry])
    base.setdefault('navigational', []).extend(brands)
    return {intent: re.compile(to_regex(words)) for intent, words in base.items() if words}

_cache = {}
def classify(keyword, language, market):
    key = (language, market)
    if key not in _cache:
        _cache[key] = build_filters(language, market)
    text = normalise(keyword)
    matched = [intent for intent, rx in _cache[key].items() if rx.search(text)]
    primary = next((p for p in PRIORITY if p in matched), 'unclassified')
    return primary, ', '.join(sorted(matched))

kw[['intent', 'all_matches']] = kw.apply(
    lambda r: pd.Series(classify(r['keyword'], r['language'], r['market'])), axis=1)
kw

## 4. Intent split per market

Same topic, four markets, very different splits. Look at both the share of keywords and the share of search volume.

In [ ]:
split = (kw.pivot_table(index='market', columns='intent', values='volume', aggfunc='sum', fill_value=0))
split_pct = (split.div(split.sum(axis=1), axis=0) * 100).round(1)
print('Share of search volume by intent (%)')
split_pct

## 5. What's still unclassified? (where to extend your filters next)

Tip 1 again: *enhance the filters with keyword alternatives, variations and localised expressions.* The words that keep showing up in unclassified keywords are your shortlist. Review them with someone who speaks the language natively before adding them.

In [ ]:
def unclassified_terms(df, top_n=15):
    out = []
    for (market, lang), g in df[df['intent'] == 'unclassified'].groupby(['market', 'language']):
        words = (g.assign(word=g['keyword'].map(lambda k: normalise(k).split()))
                   .explode('word')
                   .groupby('word')['volume'].sum()
                   .sort_values(ascending=False).head(top_n))
        out.append(pd.DataFrame({'market': market, 'language': lang,
                                 'word': words.index, 'volume': words.values}))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

kw['classified'] = kw['intent'] != 'unclassified'
kw['classified_volume'] = kw['volume'].where(kw['classified'], 0)
coverage = kw.groupby('market').agg(keywords=('keyword', 'size'),
                                    classified_pct=('classified', 'mean'),
                                    classified_volume=('classified_volume', 'sum'),
                                    volume=('volume', 'sum'))
coverage['classified_pct'] = (coverage['classified_pct'] * 100).round(1)
coverage['classified_volume_pct'] = (100 * coverage['classified_volume'] / coverage['volume']).round(1)
coverage = coverage[['keywords', 'classified_pct', 'classified_volume_pct']]
kw = kw.drop(columns=['classified', 'classified_volume'])
display(coverage) if 'display' in globals() else print(coverage)
unclassified_terms(kw)

## 6. Language traps: false friends, homographs and brand collisions

From the guide: *the same words in different languages mean different things* (English "gift" is "poison" in German), and a brand name that is also an everyday word can bloat your branded numbers in one market.

Add the words you want to review per language below. The notebook flags every keyword that contains one, with its volume, so you can check the SERP for that market before you trust the number.

In [ ]:
WATCHLIST = {
    'de': ['gift', 'fast', 'elf', 'grad', 'rat', 'kind'],
    'es': ['embarazada', 'constipado', 'carpeta', 'exito'],
    'fr': ['librairie', 'preservatif', 'blesser'],
    'en': [],
}
# Brand names that are also normal words in a market (e.g. a brand called "Mercury" or "Apple")
BRAND_COLLISIONS = {'US': ['stride'], 'GB': ['stride']}

def trap_flags(row):
    text = normalise(row['keyword']).split()
    flags = [w for w in WATCHLIST.get(row['language'], []) if normalise(w) in text]
    flags += [f'brand? {b}' for b in BRAND_COLLISIONS.get(row['market'], []) if normalise(b) in text]
    return ', '.join(flags)

kw['review_flag'] = kw.apply(trap_flags, axis=1)
kw[kw['review_flag'] != ''][['keyword', 'market', 'volume', 'intent', 'review_flag']]

## 7. Optional: compare intent across countries in Search Console

Tip 2 from the guide: pair your research with Search Console data to see whether a local market finds the same page with a different intent.

Export a **Search Console** performance report with the dimensions **query, page and country** (the Search Console API, Search Analytics for Sheets, or BigQuery bulk export all work), then upload it here. Columns needed: `query`, `page`, `country`, `clicks`, `impressions`. Map each country to the language you expect people to search in.

In [ ]:
USE_SAMPLE_GSC = True
COUNTRY_LANGUAGE = {'usa': 'en', 'gbr': 'en', 'esp': 'es', 'deu': 'de', 'fra': 'fr'}

if USE_SAMPLE_GSC:
    gsc = pd.DataFrame([
        ('how to choose running shoes', '/guides/choose-running-shoes', 'usa', 120, 4000),
        ('best running shoes', '/guides/choose-running-shoes', 'usa', 60, 9000),
        ('running shoes guide', '/guides/choose-running-shoes', 'gbr', 40, 1500),
        ('best running trainers', '/guides/choose-running-shoes', 'gbr', 35, 2600),
        ('como elegir zapatillas running', '/guides/choose-running-shoes', 'esp', 15, 900),
        ('comprar zapatillas running', '/guides/choose-running-shoes', 'esp', 9, 1200),
        ('zapatillas running precio', '/guides/choose-running-shoes', 'esp', 7, 800),
        ('laufschuhe kaufen', '/guides/choose-running-shoes', 'deu', 4, 700),
    ], columns=['query', 'page', 'country', 'clicks', 'impressions'])
else:
    gsc = pd.read_csv('gsc_query_page_country.csv')

COUNTRY_MARKET = {'usa': 'US', 'gbr': 'GB', 'esp': 'ES', 'deu': 'DE', 'fra': 'FR'}
gsc['language'] = gsc['country'].str.lower().map(COUNTRY_LANGUAGE).fillna('en')
gsc['market'] = gsc['country'].str.lower().map(COUNTRY_MARKET).fillna('')
gsc['intent'] = gsc.apply(lambda r: classify(r['query'], r['language'], r['market'])[0], axis=1)

by_country = gsc.pivot_table(index=['page', 'country'], columns='intent', values='clicks',
                             aggfunc='sum', fill_value=0)
by_country_pct = (by_country.div(by_country.sum(axis=1), axis=0) * 100).round(0)
by_country_pct

In the example, the same guide page is found through informational and commercial queries in the US and the UK, but in Spain about half of the clicks come from transactional queries, and in Germany all of them do. That's a sign the local market expects something different from this page: a localised buying section, a link to the local shop, or a separate page altogether.

## 8. Export: CSVs and ready-made regex for Data Studio or Google Sheets

The regex strings below work in a Data Studio calculated field (`CASE WHEN REGEXP_MATCH(LOWER(Keyword), "...") THEN "Transactional" ... END`) and in Google Sheets (`=REGEXMATCH(LOWER(A2), "...")`). Note that Data Studio and Sheets don't strip accents for you, so either keep both spellings in your filters or clean the keyword column first.

In [ ]:
def export_regex(language, market):
    rows = []
    for intent, rx in build_filters(language, market).items():
        # RE2 (Data Studio / Sheets) has no \b inside alternations the same way; use a simpler (^|\s) form
        pattern = rx.pattern.replace(r'(?:^|\b)', r'(^|\s)').replace(r'(?:\b|$)', r'(\s|$)').replace('(?:', '(')
        rows.append({'language': language, 'market': market, 'intent': intent, 're2_pattern': pattern})
    return rows

regex_table = pd.DataFrame([r for (lang, mkt) in kw[['language', 'market']].drop_duplicates().itertuples(index=False)
                            for r in export_regex(lang, mkt)])
kw.to_csv('keywords_with_intent.csv', index=False)
regex_table.to_csv('intent_regex_per_market.csv', index=False)
try:
    from google.colab import files
    files.download('keywords_with_intent.csv'); files.download('intent_regex_per_market.csv')
except ImportError:
    print('Saved keywords_with_intent.csv and intent_regex_per_market.csv')
regex_table.head(8)

## Where to go next

* Not sure which classification approach fits your data? Read [Search-Intent Classification with Machine Learning: The Approaches Explained](https://www.mlforseo.com/machine-learning-implementation-guides/serp-analysis/search-intent-classification-with-machine-learning-approaches/).
* Want the full picture of explicit and implicit intent? [What Is Search Intent in SEO?](https://www.mlforseo.com/machine-learning-implementation-guides/keyword-research/what-is-search-intent-in-seo-a-practical-guide-to-using-it-in-semantic-keyword-research/)
* Ready to build a full semantic keyword universe? The [Semantic ML-enabled Keyword Research course](https://academy.mlforseo.com/product/semantic-ml-enabled-keyword-research-course/) on the MLforSEO Academy.